In [3]:
RANDOM_STATE = 42
OUT_DIR = "runs"
RUN_NAME = "blte"

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from scipy.stats import randint, uniform
import matplotlib.pyplot as plt
import seaborn as sns


# ==========================
# 0. Cấu hình chung
# ==========================

DATA_PATH = r"D:\elliptic\blte\Labeled-Transactions-based-Dataset-of-Ethereum-Network-master\FinalDataset.xlsx"
# (hoặc .xlsx nếu bạn dùng bản excel -> dùng read_excel)

In [5]:
# ==========================
# 1. Load final_dataset
# ==========================
df = pd.read_excel(DATA_PATH)

print("Số dòng ban đầu:", len(df))
print("Các cột:", df.columns.tolist())

Số dòng ban đầu: 71250
Các cột: ['hash', 'nonce', 'transaction_index', 'from_address', 'to_address', 'value', 'gas', 'gas_price', 'input', 'receipt_cumulative_gas_used', 'receipt_gas_used', 'block_timestamp', 'block_number', 'block_hash', 'from_scam', 'to_scam', 'from_category', 'to_category']


In [6]:
df['rcpt_cum_gas_used'] = df['receipt_cumulative_gas_used']
df = df.drop(columns='receipt_cumulative_gas_used')

In [7]:
# ==========================
# 2. Tạo nhãn transaction-level
# ==========================
# Điền thiếu cho from_scam/to_scam (nếu có NaN)
for col in ["from_scam", "to_scam"]:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# 1 = abnormal nếu from hoặc to là scam
df["label"] = (
    (df.get("from_scam", 0) == 1) |
    (df.get("to_scam", 0) == 1)
).astype(int)

LABEL_COL = "label"

print("Phân bố nhãn (0=normal, 1=abnormal):")
print(df[LABEL_COL].value_counts())

y = df[LABEL_COL].values

Phân bố nhãn (0=normal, 1=abnormal):
label
0    57000
1    14250
Name: count, dtype: int64


In [8]:
# ==========================
# 3. Chọn feature (tránh rò rỉ label)
# ==========================

# (A) Lấy cột thời gian tách riêng (để split), KHÔNG đưa vào feature
ts_col = None
for c in ["block_timestamp", "block_number"]:
    if c in df.columns:
        ts_col = c
        break
if ts_col is None:
    raise ValueError("Không tìm thấy cột thời gian (block_timestamp hoặc block_number).")

ts_raw = df[ts_col].copy()

# Chuẩn hoá ts -> unix seconds (nếu là datetime string như ảnh của bạn)
# Nếu là số (block_number) thì giữ nguyên numeric
if ts_col == "block_timestamp":
    ts_dt = pd.to_datetime(ts_raw, errors="coerce")
    # nếu có NaT, ta ffill/bfill để không vỡ split
    ts_dt = ts_dt.fillna(method="ffill").fillna(method="bfill")
    ts_num = (ts_dt.view("int64") // 10**9).astype("int64")
else:
    ts_num = pd.to_numeric(ts_raw, errors="coerce")
    ts_num = ts_num.fillna(method="ffill").fillna(method="bfill").astype("int64")


# (B) Các cột KHÔNG dùng làm feature:
drop_id_cols = [
    # 'block_number',
    'block_timestamp',   # vẫn drop khỏi feature (đúng ý bạn)
    'txId',

    "hash",
    'transaction_index',
    "from_address",
    "to_address",
    "block_hash",
    "input"
]

drop_label_cols = [
    "from_scam", "to_scam",
    "from_category", "to_category",
    LABEL_COL
]

cols_to_drop = [c for c in drop_id_cols + drop_label_cols if c in df.columns]

feature_candidates = [c for c in df.columns if c not in cols_to_drop]
X_df = df[feature_candidates].copy()

# Giữ lại cột numeric để dùng cho ML
non_numeric = X_df.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    print("Bỏ cột không phải số:", non_numeric)
    X_df = X_df.drop(columns=non_numeric)

feature_cols = X_df.columns.tolist()
print("Feature dùng để train:", feature_cols)

X = X_df.values

# Giữ txId để align với preds từ GNN
if "txId" in df.columns:
    txid = df.loc[X_df.index, "txId"].astype(int).values
else:
    txid = np.arange(len(X_df), dtype=int)
print("txid shape:", txid.shape)

# Quan trọng: ts_num phải align đúng với X (cùng index X_df)
ts_num = ts_num.loc[X_df.index].to_numpy()


Feature dùng để train: ['nonce', 'value', 'gas', 'gas_price', 'receipt_gas_used', 'block_number', 'rcpt_cum_gas_used']
txid shape: (71250,)


C:\Users\Admin\AppData\Local\Temp\ipykernel_14548\18801085.py:21: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  ts_dt = ts_dt.fillna(method="ffill").fillna(method="bfill")
C:\Users\Admin\AppData\Local\Temp\ipykernel_14548\18801085.py:22: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  ts_num = (ts_dt.view("int64") // 10**9).astype("int64")


In [9]:
# ==========================
# 3b. Kiểm tra tương quan feature với label
# ==========================

# Ở đây ta dùng chính X_df (sau khi đã drop ID/label, xử lý timestamp, bỏ non-numeric)
# và cột nhãn LABEL_COL trong df.

# Gộp X_df (feature) với cột label thành một DataFrame chung
corr_df = pd.concat(
    [
        X_df.reset_index(drop=True),
        df[LABEL_COL].reset_index(drop=True)
    ],
    axis=1
)

# Tính ma trận tương quan Pearson
corr_matrix = corr_df.corr()

# Lấy vector tương quan của từng feature với label
corr_with_label = corr_matrix[LABEL_COL].drop(labels=[LABEL_COL])  # bỏ chính label

# Giá trị tuyệt đối để xem mức độ mạnh/yếu
corr_with_label_abs = corr_with_label.abs().sort_values(ascending=False)

print("\nTop 30 feature có |corr| lớn nhất với label:")
print(corr_with_label_abs.head(30))

# Nếu muốn xem cả dấu và |corr| dưới dạng bảng:
corr_table = pd.DataFrame({
    "corr": corr_with_label,
    "abs_corr": corr_with_label_abs
}).sort_values("abs_corr", ascending=False)

corr_table.head(30)



Top 30 feature có |corr| lớn nhất với label:
block_number         0.442214
nonce                0.120640
rcpt_cum_gas_used    0.110460
receipt_gas_used     0.105110
gas_price            0.048455
gas                  0.022670
value                0.015716
Name: label, dtype: float64


,corr,abs_corr
block_number,0.442214,0.442214
nonce,-0.120640,0.120640
rcpt_cum_gas_used,0.110460,0.110460
receipt_gas_used,0.105110,0.105110
gas_price,-0.048455,0.048455
gas,0.022670,0.022670
value,-0.015716,0.015716


In [10]:
# ==========================
# 4. Chia train / val / test THEO THỜI GIAN (70/15/15) - KHÔNG XÁO TRỘN
# ==========================
# Ý tưởng giống Elliptic (Tree): chia theo "time bucket"
# tất cả transaction thuộc cùng 1 time sẽ nằm trong cùng 1 split.

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
assert abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) < 1e-9

# Ở cell 3 bạn đã tạo:
#   ts_col  : tên cột thời gian (block_timestamp hoặc block_number)
#   ts_num  : numpy array unix seconds hoặc block_number, đã align với X/y/txid
# Nên ở đây KHÔNG cần đọc df/parse lại nữa.

unique_ts = np.sort(np.unique(ts_num))
n_ts = len(unique_ts)

print(f"ts_col = {ts_col} | số time buckets = {n_ts}")
print("time đầu/cuối:", unique_ts[:5], "...", unique_ts[-5:])

# Cắt theo tỉ lệ time buckets (không phải theo số dòng)
train_cut = int(np.floor(TRAIN_RATIO * n_ts))
val_cut   = int(np.floor((TRAIN_RATIO + VAL_RATIO) * n_ts))

# Đảm bảo mỗi split có ít nhất 1 time bucket
train_cut = max(train_cut, 1)
val_cut   = max(val_cut, train_cut + 1)
val_cut   = min(val_cut, n_ts - 1)

train_ts = unique_ts[:train_cut]
val_ts   = unique_ts[train_cut:val_cut]
test_ts  = unique_ts[val_cut:]

train_mask = np.isin(ts_num, train_ts)
val_mask   = np.isin(ts_num, val_ts)
test_mask  = np.isin(ts_num, test_ts)

# Sanity check: không overlap & cover hết
assert not np.any(train_mask & val_mask)
assert not np.any(train_mask & test_mask)
assert not np.any(val_mask & test_mask)
assert np.all(train_mask | val_mask | test_mask)

X_train, y_train, txid_train = X[train_mask], y[train_mask], txid[train_mask]
X_val,   y_val,   txid_val   = X[val_mask],   y[val_mask],   txid[val_mask]
X_test,  y_test,  txid_test  = X[test_mask],  y[test_mask],  txid[test_mask]

def show_stats(name, yy):
    counts = np.bincount(yy)
    n0 = counts[0] if len(counts) > 0 else 0
    n1 = counts[1] if len(counts) > 1 else 0
    ratio = n1 / (n0 + n1) if (n0 + n1) > 0 else 0
    print(f"{name:5s}: 0 = {n0:6d}, 1 = {n1:6d}, scam_ratio = {ratio:.6f}")

print("\n=== Phân bố nhãn sau khi chia (theo time) ===")
show_stats("ALL",   y)
show_stats("Train", y_train)
show_stats("Val",   y_val)
show_stats("Test",  y_test)

print("\nKích thước:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val  :", X_val.shape,   "y_val  :", y_val.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)


ts_col = block_timestamp | số time buckets = 28683
time đầu/cuối: [1508131613 1508131729 1508131759 1508131783 1508131817] ... [1570405665 1570406197 1570406223 1570406280 1570406377]

=== Phân bố nhãn sau khi chia (theo time) ===
ALL  : 0 =  57000, 1 =  14250, scam_ratio = 0.200000
Train: 0 =  42538, 1 =   3021, scam_ratio = 0.066310
Val  : 0 =   7200, 1 =    699, scam_ratio = 0.088492
Test : 0 =   7262, 1 =  10530, scam_ratio = 0.591839

Kích thước:
X_train: (45559, 7) y_train: (45559,)
X_val  : (7899, 7) y_val  : (7899,)
X_test : (17792, 7) y_test : (17792,)


In [11]:
print("Feature dùng để train:", feature_cols)
print("Các cột nghi ngờ (scam/category/label):")
print([c for c in feature_cols
       if any(k in c.lower() for k in ["scam", "category", "label"])])

Feature dùng để train: ['nonce', 'value', 'gas', 'gas_price', 'receipt_gas_used', 'block_number', 'rcpt_cum_gas_used']
Các cột nghi ngờ (scam/category/label):
[]


In [12]:
# ==========================
# 5. Chuẩn hóa feature
# ==========================
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

In [13]:
# =========================================================
# 6.x UNSUPERVISED MODELS (AutoML style): Isolation Forest + AutoEncoder
# Flow:
#   - Fit trên TRAIN (ưu tiên normal class y=0)
#   - Tune hyperparams + threshold bằng VAL (dùng F1)
#   - Refit final trên TRAIN+VAL
#   - Test trên TEST
# =========================================================

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.ensemble import IsolationForest
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix, roc_auc_score
from scipy.stats import randint, uniform

# ====== Nếu bạn đã có sample_param ở cell trước thì có thể bỏ hàm này ======
def sample_param(dist, rng):
    if hasattr(dist, "rvs"):
        return dist.rvs(random_state=rng)
    dist = list(dist)
    return dist[rng.randint(0, len(dist))]

In [14]:
# =========================================================
# Helpers chung cho unsupervised anomaly score
# =========================================================
def best_threshold_by_f1(y_true, anomaly_scores, n_grid=200):
    """
    anomaly_scores: score càng lớn càng bất thường (class 1)
    Chọn threshold tối ưu trên VAL theo F1(binary, pos_label=1)
    """
    y_true = np.asarray(y_true).astype(int)
    s = np.asarray(anomaly_scores).ravel()

    # grid threshold theo quantile để ổn định
    qs = np.linspace(0.01, 0.99, n_grid)
    thrs = np.unique(np.quantile(s, qs))

    best = {
        "threshold": None,
        "f1": -1.0,
        "acc": None,
        "f1_macro": None,
        "f1_micro": None
    }

    for t in thrs:
        y_pred = (s >= t).astype(int)
        f1_bin = f1_score(y_true, y_pred, zero_division=0)
        if f1_bin > best["f1"]:
            best["threshold"] = float(t)
            best["f1"] = float(f1_bin)
            best["acc"] = float(accuracy_score(y_true, y_pred))
            best["f1_macro"] = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
            best["f1_micro"] = float(f1_score(y_true, y_pred, average="micro", zero_division=0))

    return best

def eval_detector_on_test(name, model, X_test, y_test):
    """
    model cần có:
      - predict(X)
      - predict_proba(X) hoặc decision_function(X)
    """
    print(f"\n===== {name} trên TEST =====")
    y_pred = model.predict(X_test)

    y_score = None
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)
        if proba.ndim == 2 and proba.shape[1] >= 2:
            y_score = proba[:, 1]
        else:
            y_score = np.asarray(proba).ravel()
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)

    acc      = accuracy_score(y_test, y_pred)
    f1_scam  = f1_score(y_test, y_pred, zero_division=0)
    f1_micro = f1_score(y_test, y_pred, average="micro", zero_division=0)
    f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)

    if y_score is not None and len(np.unique(y_test)) >= 2:
        try:
            auc = roc_auc_score(y_test, y_score)
        except Exception:
            auc = np.nan
    else:
        auc = np.nan

    print(f"Accuracy: {acc:.6f}")
    print(f"F1 (abnormal=1): {f1_scam:.6f}")
    print(f"F1 micro: {f1_micro:.6f}")
    print(f"F1 macro: {f1_macro:.6f}")
    print(f"ROC-AUC: {auc:.6f}" if np.isfinite(auc) else "ROC-AUC: nan")

    print("\nclassification_report:")
    print(classification_report(y_test, y_pred, digits=6, zero_division=0))

    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))


In [15]:
# =========================================================
# 1) Isolation Forest wrapper + AutoML (train on normal only)
# =========================================================
class IForestDetector(BaseEstimator, ClassifierMixin):
    """
    Wrapper để dùng như classifier:
      - anomaly score = -score_samples(X) (càng lớn càng bất thường)
      - threshold được tune trên VAL
    """
    def __init__(self, random_state=42, n_estimators=200, max_samples='auto',
                 max_features=1.0, bootstrap=False, threshold=None):
        self.random_state = random_state
        self.n_estimators = n_estimators
        self.max_samples = max_samples
        self.max_features = max_features
        self.bootstrap = bootstrap
        self.threshold = threshold

    def _make_model(self):
        return IsolationForest(
            n_estimators=self.n_estimators,
            max_samples=self.max_samples,
            max_features=self.max_features,
            bootstrap=self.bootstrap,
            contamination='auto',   # threshold sẽ do ta tự tune
            random_state=self.random_state,
            n_jobs=-1
        )

    def fit(self, X, y=None):
        # nếu y có sẵn -> chỉ fit trên normal (0)
        if y is not None:
            X_fit = X[np.asarray(y) == 0]
            if len(X_fit) == 0:
                X_fit = X
        else:
            X_fit = X

        self.model_ = self._make_model()
        self.model_.fit(X_fit)
        return self

    def anomaly_score(self, X):
        # score lớn = bất thường
        return -self.model_.score_samples(X)

    def decision_function(self, X):
        # dùng anomaly score làm continuous score cho ROC-AUC
        return self.anomaly_score(X)

    def predict_proba(self, X):
        s = self.anomaly_score(X)
        # scale min-max để giả lập proba (phục vụ ensemble soft/ROC-AUC)
        s_min, s_max = np.min(s), np.max(s)
        if s_max - s_min < 1e-12:
            p1 = np.full_like(s, 0.5, dtype=float)
        else:
            p1 = (s - s_min) / (s_max - s_min)
        p1 = np.clip(p1, 0.0, 1.0)
        return np.vstack([1 - p1, p1]).T

    def predict(self, X):
        if self.threshold is None:
            raise ValueError("threshold chưa được set/tune.")
        s = self.anomaly_score(X)
        return (s >= self.threshold).astype(int)

def random_search_iforest_unsup(
    X_train, y_train, X_val, y_val,
    n_iter=20, random_state=42
):
    print("\n===== Random search cho IsolationForest (unsupervised, tune bằng VAL) =====")
    rng = np.random.RandomState(random_state)

    # search space (AutoML style)
    param_dist = {
        "n_estimators": randint(100, 500),
        "max_samples": uniform(0.4, 0.6),   # [0.4, 1.0]
        "max_features": uniform(0.5, 0.5),  # [0.5, 1.0]
        "bootstrap": [False, True],
    }

    best = {
        "f1": -1.0,
        "threshold": None,
        "params": None
    }

    for i in range(n_iter):
        params = {k: sample_param(v, rng) for k, v in param_dist.items()}

        model = IForestDetector(
            random_state=random_state,
            n_estimators=int(params["n_estimators"]),
            max_samples=float(params["max_samples"]),
            max_features=float(params["max_features"]),
            bootstrap=bool(params["bootstrap"]),
            threshold=None
        )
        model.fit(X_train, y_train)  # chỉ fit trên normal

        val_scores = model.anomaly_score(X_val)
        th_info = best_threshold_by_f1(y_val, val_scores, n_grid=200)

        print(
            f"Iter {i+1:02d}/{n_iter}: "
            f"F1(val)={th_info['f1']:.6f}, thr={th_info['threshold']:.6f}, params={params}"
        )

        if th_info["f1"] > best["f1"]:
            best["f1"] = th_info["f1"]
            best["threshold"] = th_info["threshold"]
            best["params"] = params

    print("\n>>> IsolationForest – best F1(val) =", f"{best['f1']:.6f}")
    print("Best params:", best["params"])
    print("Best threshold:", best["threshold"])

    # -------- Refit final trên TRAIN+VAL (normal only) --------
    X_train_full = np.vstack([X_train, X_val])
    y_train_full = np.concatenate([y_train, y_val])

    best_if = IForestDetector(
        random_state=random_state,
        n_estimators=int(best["params"]["n_estimators"]),
        max_samples=float(best["params"]["max_samples"]),
        max_features=float(best["params"]["max_features"]),
        bootstrap=bool(best["params"]["bootstrap"]),
        threshold=float(best["threshold"])
    )
    best_if.fit(X_train_full, y_train_full)

    return best_if, best

In [16]:
# Chạy AutoML cho IsolationForest
best_iforest, best_iforest_info = random_search_iforest_unsup(
    X_train_scaled, y_train,
    X_val_scaled,   y_val,
    n_iter=20,
    random_state=RANDOM_STATE
)


===== Random search cho IsolationForest (unsupervised, tune bằng VAL) =====
Iter 01/20: F1(val)=0.372591, thr=0.447120, params={'n_estimators': 202, 'max_samples': np.float64(0.8779257921161396), 'max_features': np.float64(0.5917173949330818), 'bootstrap': True}
Iter 02/20: F1(val)=0.291566, thr=0.417442, params={'n_estimators': 288, 'max_samples': np.float64(0.7581100947678923), 'max_features': np.float64(0.7229163764267956), 'bootstrap': False}
Iter 03/20: F1(val)=0.306769, thr=0.437666, params={'n_estimators': 430, 'max_samples': np.float64(0.6755493351795203), 'max_features': np.float64(0.6668543055695109), 'bootstrap': True}
Iter 04/20: F1(val)=0.377336, thr=0.458662, params={'n_estimators': 251, 'max_samples': np.float64(0.7905330837693118), 'max_features': np.float64(0.5282057895135501), 'bootstrap': True}
Iter 05/20: F1(val)=0.272266, thr=0.411415, params={'n_estimators': 393, 'max_samples': np.float64(0.4004672595046086), 'max_features': np.float64(0.9961057796456088), 'boots

In [17]:
# =========================================================
# 3) Đánh giá unsupervised models trên TEST
# =========================================================
eval_detector_on_test("IsolationForest", best_iforest, X_test_scaled, y_test)


===== IsolationForest trên TEST =====
Accuracy: 0.755621
F1 (abnormal=1): 0.772380
F1 micro: 0.755621
F1 macro: 0.754288
ROC-AUC: 0.735757

classification_report:
              precision    recall  f1-score   support

           0   0.658026  0.835445  0.736197      7262
           1   0.860593  0.700570  0.772380     10530

    accuracy                       0.755621     17792
   macro avg   0.759309  0.768007  0.754288     17792
weighted avg   0.777913  0.755621  0.757611     17792

Confusion matrix:
[[6067 1195]
 [3153 7377]]


In [18]:
# =========================================================
# 2) AutoEncoder wrapper + AutoML (train on normal only)
# =========================================================
import warnings
warnings.filterwarnings("ignore")

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
except Exception as e:
    raise ImportError(
        "Cần tensorflow/keras để chạy AutoEncoder. "
        "Nếu bạn muốn, tôi có thể viết bản PyTorch tương đương."
    ) from e

# để reproducible hơn
tf.keras.utils.set_random_seed(RANDOM_STATE)

class AEDetector(BaseEstimator, ClassifierMixin):
    """
    AutoEncoder anomaly detector:
      - fit trên normal class (0)
      - anomaly score = reconstruction error (MSE)
      - threshold tune trên VAL
    """
    def __init__(self,
                 input_dim=None,
                 hidden_dims=(64, 32),
                 latent_dim=16,
                 activation="relu",
                 dropout=0.0,
                 learning_rate=1e-3,
                 batch_size=256,
                 epochs=30,
                 threshold=None,
                 random_state=42,
                 verbose=0):
        self.input_dim = input_dim
        self.hidden_dims = hidden_dims
        self.latent_dim = latent_dim
        self.activation = activation
        self.dropout = dropout
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epochs = epochs
        self.threshold = threshold
        self.random_state = random_state
        self.verbose = verbose

    def _build_model(self, input_dim):
        inp = keras.Input(shape=(input_dim,))
        x = inp

        # Encoder
        for h in self.hidden_dims:
            x = layers.Dense(h, activation=self.activation)(x)
            if self.dropout > 0:
                x = layers.Dropout(self.dropout)(x)

        z = layers.Dense(self.latent_dim, activation=self.activation, name="latent")(x)

        # Decoder (đối xứng)
        x = z
        for h in reversed(self.hidden_dims):
            x = layers.Dense(h, activation=self.activation)(x)
        out = layers.Dense(input_dim, activation="linear")(x)

        model = keras.Model(inp, out, name="ae_detector")
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=self.learning_rate),
            loss="mse"
        )
        return model

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=np.float32)
        if y is not None:
            X_fit = X[np.asarray(y) == 0]  # chỉ normal
            if len(X_fit) == 0:
                X_fit = X
        else:
            X_fit = X

        input_dim = X.shape[1] if self.input_dim is None else self.input_dim

        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(self.random_state)

        self.model_ = self._build_model(input_dim)

        callbacks = [
            keras.callbacks.EarlyStopping(
                monitor="loss",
                patience=5,
                restore_best_weights=True
            )
        ]

        self.history_ = self.model_.fit(
            X_fit, X_fit,
            epochs=int(self.epochs),
            batch_size=int(self.batch_size),
            shuffle=True,
            verbose=self.verbose,
            callbacks=callbacks
        )
        return self

    def anomaly_score(self, X):
        X = np.asarray(X, dtype=np.float32)
        X_hat = self.model_.predict(X, batch_size=int(self.batch_size), verbose=0)
        err = np.mean((X - X_hat) ** 2, axis=1)  # MSE theo sample
        return err

    def decision_function(self, X):
        return self.anomaly_score(X)

    def predict_proba(self, X):
        s = self.anomaly_score(X)
        s_min, s_max = np.min(s), np.max(s)
        if s_max - s_min < 1e-12:
            p1 = np.full_like(s, 0.5, dtype=float)
        else:
            p1 = (s - s_min) / (s_max - s_min)
        p1 = np.clip(p1, 0.0, 1.0)
        return np.vstack([1 - p1, p1]).T

    def predict(self, X):
        if self.threshold is None:
            raise ValueError("threshold chưa được set/tune.")
        s = self.anomaly_score(X)
        return (s >= self.threshold).astype(int)

def random_search_ae_unsup(
    X_train, y_train, X_val, y_val,
    n_iter=12, random_state=42
):
    print("\n===== Random search cho AutoEncoder (unsupervised, tune bằng VAL) =====")
    rng = np.random.RandomState(random_state)

    # search space (AutoML style)
    param_dist = {
        "hidden_dims": [(64, 32), (128, 64), (128, 64, 32), (256, 128, 64)],
        "latent_dim": randint(8, 65),
        "activation": ["relu", "elu"],
        "dropout": [0.0, 0.1, 0.2],
        "learning_rate": [1e-3, 5e-4, 2e-4],
        "batch_size": [128, 256, 512],
        "epochs": randint(20, 61),
    }

    best = {
        "f1": -1.0,
        "threshold": None,
        "params": None
    }

    for i in range(n_iter):
        params = {k: sample_param(v, rng) for k, v in param_dist.items()}

        model = AEDetector(
            input_dim=X_train.shape[1],
            hidden_dims=tuple(params["hidden_dims"]),
            latent_dim=int(params["latent_dim"]),
            activation=str(params["activation"]),
            dropout=float(params["dropout"]),
            learning_rate=float(params["learning_rate"]),
            batch_size=int(params["batch_size"]),
            epochs=int(params["epochs"]),
            threshold=None,
            random_state=random_state + i,  # tránh trùng y hệt seed mọi vòng
            verbose=0
        )

        model.fit(X_train, y_train)  # chỉ normal
        val_scores = model.anomaly_score(X_val)
        th_info = best_threshold_by_f1(y_val, val_scores, n_grid=200)

        print(
            f"Iter {i+1:02d}/{n_iter}: "
            f"F1(val)={th_info['f1']:.6f}, thr={th_info['threshold']:.6f}, params={params}"
        )

        if th_info["f1"] > best["f1"]:
            best["f1"] = th_info["f1"]
            best["threshold"] = th_info["threshold"]
            best["params"] = params

    print("\n>>> AutoEncoder – best F1(val) =", f"{best['f1']:.6f}")
    print("Best params:", best["params"])
    print("Best threshold:", best["threshold"])

    # -------- Refit final trên TRAIN+VAL (normal only) --------
    X_train_full = np.vstack([X_train, X_val])
    y_train_full = np.concatenate([y_train, y_val])

    bp = best["params"]
    best_ae = AEDetector(
        input_dim=X_train_full.shape[1],
        hidden_dims=tuple(bp["hidden_dims"]),
        latent_dim=int(bp["latent_dim"]),
        activation=str(bp["activation"]),
        dropout=float(bp["dropout"]),
        learning_rate=float(bp["learning_rate"]),
        batch_size=int(bp["batch_size"]),
        epochs=int(bp["epochs"]),
        threshold=float(best["threshold"]),
        random_state=random_state,
        verbose=0
    )
    best_ae.fit(X_train_full, y_train_full)

    return best_ae, best

# Chạy AutoML cho AutoEncoder
best_ae, best_ae_info = random_search_ae_unsup(
    X_train_scaled, y_train,
    X_val_scaled,   y_val,
    n_iter=12,   # AE tốn thời gian hơn IF, bạn tăng lên nếu muốn
    random_state=RANDOM_STATE
)


===== Random search cho AutoEncoder (unsupervised, tune bằng VAL) =====

Iter 01/12: F1(val)=0.366822, thr=0.034316, params={'hidden_dims': (128, 64, 32), 'latent_dim': 59, 'activation': 'relu', 'dropout': 0.2, 'learning_rate': 0.0002, 'batch_size': 128, 'epochs': 40}
Iter 02/12: F1(val)=0.334009, thr=0.056577, params={'hidden_dims': (128, 64, 32), 'latent_dim': 26, 'activation': 'relu', 'dropout': 0.2, 'learning_rate': 0.0002, 'batch_size': 128, 'epochs': 55}
Iter 03/12: F1(val)=0.300395, thr=0.068406, params={'hidden_dims': (256, 128, 64), 'latent_dim': 31, 'activation': 'relu', 'dropout': 0.1, 'learning_rate': 0.001, 'batch_size': 256, 'epochs': 43}
Iter 04/12: F1(val)=0.182159, thr=0.006277, params={'hidden_dims': (256, 128, 64), 'latent_dim': 37, 'activation': 'elu', 'dropout': 0.1, 'learning_rate': 0.001, 'batch_size': 128, 'epochs': 31}
Iter 05/12: F1(val)=0.169666, thr=0.009380, params={'hidden_dims': (128, 64), 'latent_dim': 29, 'activation': 'relu', 'dropout': 0.0, 'learning

In [19]:
eval_detector_on_test("AutoEncoder",     best_ae,     X_test_scaled, y_test)


===== AutoEncoder trên TEST =====
Accuracy: 0.783217
F1 (abnormal=1): 0.833943
F1 micro: 0.783217
F1 macro: 0.760906
ROC-AUC: 0.754967

classification_report:
              precision    recall  f1-score   support

           0   0.834151  0.585238  0.687869      7262
           1   0.762779  0.919753  0.833943     10530

    accuracy                       0.783217     17792
   macro avg   0.798465  0.752496  0.760906     17792
weighted avg   0.791910  0.783217  0.774322     17792

Confusion matrix:
[[4250 3012]
 [ 845 9685]]
